In [ ]:
#-- Load libraries
library(readr)
library(dplyr)
library(data.table)
library(ggplot2)
library(lubridate)
library(purrr)
library(stringr)
library(tidyverse)
library(openxlsx)
library(nnet)
library(readxl)
library(tidyr)
library(brant)

In [ ]:
#-- Load outcomes
outcome <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_Outcomes_90days_noBIP_noPsychotic.csv") %>%
  mutate(
    FirstDispense.treatment = as.Date(FirstDispense.treatment),
    ExpectedEndDate.treatment = as.Date(ExpectedEndDate.treatment),
    FirstDispense.treatment.episode = as.Date(FirstDispense.treatment.episode),
    ExpectedEndDate.treatment.episode = as.Date(ExpectedEndDate.treatment.episode),
    FirstDispense.other.episode = as.Date(FirstDispense.other.episode),
    ExpectedEndDate.other.episode = as.Date(ExpectedEndDate.other.episode),
    Incidence = as.Date(Incidence)
  )
#-- Other outcomes
combo <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Combination_Therapy_noBIP_noPsychotic.csv") %>% dplyr::select(person_id, Case)
aug <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Augmentation_Therapy_noBIP_noPsychotic.csv") %>% dplyr::select(person_id, Case)

In [ ]:
#-- First switch
first_switch <- outcome %>%
  filter(FinalClassification.treatment == "Switching") %>%
  group_by(person_id) %>%
  arrange(Incidence, FirstDispense.treatment) %>%
  slice(1) %>%
  ungroup()

#-- First continuation (among non-switchers)
C <- outcome %>%
  filter(!person_id %in% first_switch$person_id) %>%
  filter(FinalClassification.treatment == "Continuation") %>%
  group_by(person_id) %>%
  arrange(Incidence, FirstDispense.treatment) %>%
  slice(1) %>%
  ungroup()

C_clean <- C %>%
  filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
  filter(!person_id %in% combo$person_id[combo$Case == 1])
    

#-- First discontinuation (among non-switchers AND non-continuers)
ED <- outcome %>%
  filter(!person_id %in% first_switch$person_id) %>%
  filter(!person_id %in% C_clean$person_id) %>%
  filter(FinalClassification.treatment == "Discontinuation") %>%
  group_by(person_id) %>%
  arrange(Incidence, FirstDispense.treatment) %>%
  slice(1) %>%
  ungroup()

#-- Input data for switching vs. non-switching
dat <- rbind(first_switch, C_clean, ED) %>%
    dplyr::select(person_id, FinalClassification.treatment) %>%
    distinct()


In [ ]:
final <- dat %>%
  full_join(aug, by = "person_id") %>%
  mutate(aug_status = case_when(
    Case == 1 ~ 1,
    Case == 0 & FinalClassification.treatment == "Continuation" ~ 0,
    TRUE ~ NA_integer_
  )) %>%
  dplyr::select(-Case)

cat("After aug join:", nrow(final), "\n")

#-- Define switch (no aug for ordinal analyses)
final <- final %>%
    mutate(aug_switch_cont = case_when(
      aug_status == 1                                ~ "Augmentation",
      FinalClassification.treatment == "Switching"   ~ "Switching",
      FinalClassification.treatment == "Continuation" ~ "Continuation",
      TRUE ~ NA_character_
    ))


In [ ]:
#-- Load PGS
pgs <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/PGS/all_traits_PGS.csv")
colnames(pgs)

#-- Read in predicted ancestries
workspace_bucket <- Sys.getenv("WORKSPACE_BUCKET")
anc_file_path <- file.path(workspace_bucket, "ancestry/ancestry_preds.tsv")
anc_df <- read_tsv(pipe(sprintf("gsutil cat %s", anc_file_path)))

#-- Extract PCs
anc_df_clean <- anc_df %>%
  mutate(pca_clean = str_remove_all(pca_features, "\\[|\\]")) 
anc_df_clean2 <- anc_df_clean %>%
  separate(pca_clean, into = paste0("PC", 1:16), sep = ",\\s*", convert = TRUE)
pc <- anc_df_clean2 %>% 
    dplyr::select(research_id, ancestry_pred, PC1:PC16)

# Standardize each PGS using EUR means and SDs
pgs_pc <- pgs %>% left_join(pc, by = c("IID" = "research_id"))
eur_pgs_pc <- pgs_pc %>% 
    filter(ancestry_pred == "eur")

# Define all PGS columns to standardize
pgs_cols <- c("ADHD_01_PGS", "Migraine_01_PGS", "Insomnia_01_PGS", "BMI_LOO_PGS", "BIP_2025_PGS",
             "MDD_04_PGS", "SCZ_03_PGS")

for (col in pgs_cols) {
  meu <- mean(eur_pgs_pc[[col]], na.rm = TRUE)
  sd <- sd(eur_pgs_pc[[col]], na.rm = TRUE)
  
  new_col <- paste0(col, "_std")
  pgs_pc[[new_col]] <- (pgs_pc[[col]] - meu) / sd
}

In [ ]:
#== Join with phenotypes
#-- Load Phenotypes
pheno <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv")
pheno <- pheno %>% 
    mutate(Income_quantitative =
          case_when(Income == ">200k" ~ 8,
                   Income == "150-200k" ~ 7,
                   Income == "100-150k" ~ 6,
                   Income == "75-100k" ~ 5,
                   Income == "50-75k" ~ 4,
                   Income == "35-50k" ~ 3,
                   Income == "25-35k" ~ 2,
                   Income == "<=25k" ~ 1,
                   TRUE ~ NA_integer_))

first_dispense <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/First_Dispense.csv")


final <- final %>%
  left_join(first_dispense, by = "person_id") %>%
  left_join(pheno, by = "person_id") %>%
  mutate(
    age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)
  ) %>%
    left_join(pgs_pc, by = c("person_id" = "IID"))

In [ ]:
model_function <- function(input_data, independent, outcome) {
  
  independent_sym <- rlang::sym(independent)
  outcome_sym <- rlang::sym(outcome)
  
  print(paste("Running model for:", independent, "->", outcome))
  
  #-- Filter for complete cases (include all 16 PCs for residualisation)
  input_data <- input_data %>%
    dplyr::select(age_at_first_AD, sex_at_birth, Income_quantitative, Education_level,
           PC1, PC2, PC3, PC4, PC5, PC6, PC7, PC8, PC9, PC10, PC11, PC12, PC13, PC14, PC15, PC16,
           !!independent_sym, !!outcome_sym) %>%
    filter(complete.cases(.))
  
  #-- Residualise PGS on all 16 global PCs
  pc_formula <- as.formula(paste(independent, "~ PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10 +
                                  PC11 + PC12 + PC13 + PC14 + PC15 + PC16"))
  input_data$pgs_residualised <- residuals(lm(pc_formula, data = input_data))
  
  #-- Calculate prevalence
  switch_counts <- input_data %>%
    group_by(!!outcome_sym) %>%
    summarise(n = n(), .groups = "drop")
  
  n_cases <- switch_counts %>% filter(!!outcome_sym == 1) %>% pull(n)
  n_controls <- switch_counts %>% filter(!!outcome_sym == 0) %>% pull(n)
  
  if (length(n_cases) == 0) n_cases <- 0
  if (length(n_controls) == 0) n_controls <- 0
  
  n_totals <- n_cases + n_controls
  
  #-- Skip if insufficient sample size
  if (n_controls < 20 | n_cases < 20) {
    return(data.frame(
      message = "Insufficient sample size",
      N_Cases = n_cases,
      N_Controls = n_controls,
      N_Total = n_totals
    ))
  }
  
  #-- GLM Logistic Regression — no PCs in outcome model
  glm_formula <- as.formula(paste0(
    outcome,
    " ~ age_at_first_AD + sex_at_birth + Income_quantitative + Education_level + pgs_residualised"
  ))
  
  glm_model <- glm(glm_formula, data = input_data, family = binomial(link = "logit"))
  
  conf_intervals <- suppressMessages(confint(glm_model))
  
  #-- Rename pgs_residualised back to original PGS name
  coef_names <- names(coef(glm_model))
  coef_names[coef_names == "pgs_residualised"] <- independent
  rownames(conf_intervals)[rownames(conf_intervals) == "pgs_residualised"] <- independent
  
  glm_results <- data.frame(
    variable   = coef_names,
    coef       = coef(glm_model),
    odds_ratio = exp(coef(glm_model)),
    se         = summary(glm_model)$coefficients[, "Std. Error"],
    z          = summary(glm_model)$coefficients[, "z value"],
    p_value    = summary(glm_model)$coefficients[, "Pr(>|z|)"],
    lower_95   = exp(conf_intervals[, 1]),
    upper_95   = exp(conf_intervals[, 2]),
    N_Total    = n_totals,
    N_Cases    = n_cases,
    N_Controls = n_controls,
    stringsAsFactors = FALSE,
    row.names = NULL
  )
  
  return(glm_results)
}

In [ ]:
model_function_ordinal <- function(input_data, independent) {
  
  independent_sym <- rlang::sym(independent)
  
  print(paste("Running ordinal model for:", independent))
  
  #-- Filter for complete cases (include all 16 PCs for residualisation)
  input_data <- input_data %>%
    dplyr::select(aug_switch_cont, age_at_first_AD, sex_at_birth, 
                  Income_quantitative, Education_level,
                  PC1, PC2, PC3, PC4, PC5, PC6, PC7, PC8, PC9, PC10, 
                  PC11, PC12, PC13, PC14, PC15, PC16,
                  !!independent_sym) %>%
    filter(complete.cases(.))
  
  if (nrow(input_data) == 0) {
    message("No complete cases remaining")
    return("Incomplete")
  }
  
  #-- Residualise PGS on all 16 global PCs
  pc_formula <- as.formula(paste(independent, "~ PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10 +
                                  PC11 + PC12 + PC13 + PC14 + PC15 + PC16"))
  input_data$pgs_residualised <- residuals(lm(pc_formula, data = input_data))
  
  #-- Set ordered factor: Continuation < Switching < Augmentation
  input_data$outcome_ord <- factor(
    input_data$aug_switch_cont,
    levels  = c("Continuation", "Switching", "Augmentation"),
    ordered = TRUE
  )
  
  #-- Outcome counts
  outcome_counts <- input_data %>%
    group_by(outcome_ord) %>%
    summarise(n = n(), .groups = "drop")
  
  n_continuation <- outcome_counts %>% filter(outcome_ord == "Continuation") %>% pull(n)
  n_switching    <- outcome_counts %>% filter(outcome_ord == "Switching")    %>% pull(n)
  n_augmentation <- outcome_counts %>% filter(outcome_ord == "Augmentation") %>% pull(n)
  
  if (length(n_continuation) == 0) n_continuation <- 0
  if (length(n_switching) == 0)    n_switching    <- 0
  if (length(n_augmentation) == 0) n_augmentation <- 0
  
  n_total <- nrow(input_data)
  
  if (n_continuation < 20 | n_switching < 20 | n_augmentation < 20) {
    message(sprintf("Insufficient cases: Cont=%d, Switch=%d, Aug=%d", 
                    n_continuation, n_switching, n_augmentation))
    return("Incomplete")
  }
  
  #-- Ordinal regression formula — no PCs in outcome model
  ord_formula <- as.formula(
    "outcome_ord ~ age_at_first_AD + sex_at_birth + Income_quantitative + Education_level + pgs_residualised"
  )
  
  ord_model <- tryCatch(
    MASS::polr(ord_formula, data = input_data, Hess = TRUE, method = "logistic"),
    error = function(e) {
      message("polr failed: ", e$message)
      return(NULL)
    }
  )
  
  if (is.null(ord_model)) return("Incomplete")
  
  coef_summary <- summary(ord_model)$coefficients
  
  betas  <- coef_summary[, "Value"]
  ses    <- coef_summary[, "Std. Error"]
  t_vals <- coef_summary[, "t value"]
  p_vals <- 2 * pnorm(abs(t_vals), lower.tail = FALSE)
  
  lower <- betas - 1.96 * ses
  upper <- betas + 1.96 * ses
  
  zeta_names <- names(ord_model$zeta)
  is_coef    <- !(rownames(coef_summary) %in% zeta_names)
  
  var_names <- rownames(coef_summary)
  var_names[var_names == "pgs_residualised"] <- independent
  
  results <- data.frame(
    variable         = var_names,
    term_type        = ifelse(is_coef, "coefficient", "intercept"),
    coef             = betas,
    odds_ratio       = exp(betas),
    se               = ses,
    t_value          = t_vals,
    p_value          = p_vals,
    lower_95         = exp(lower),
    upper_95         = exp(upper),
    N_Total          = n_total,
    N_Continuation   = n_continuation,
    N_Switching      = n_switching,
    N_Augmentation   = n_augmentation,
    stringsAsFactors = FALSE,
    row.names        = NULL
  )
  
  return(results)
}

In [ ]:
#-- Select phenotypes to be associated with 
predictors <- c("ADHD_01_PGS_std", "Migraine_01_PGS_std","Insomnia_01_PGS_std", "BMI_LOO_PGS_std", "BIP_2025_PGS_std",
             "MDD_04_PGS_std", "SCZ_03_PGS_std")

#-- Setup
population_groups <- c("afr", "eur", "amr")
clinical_groups <- c("Full", "MDD", "noMDD", "ANX", "CIDI-MDD", "CIDI-noMDD")
outcomes <- c("aug_status")

#-- Main loop with better structure
all_list <- map(clinical_groups, function(clinical) {
  
  cat("\n=== Processing:", clinical, "===\n")
  
  #-- Filter once for clinical group
    if (clinical == "MDD") {
      input_data <- final %>%
        filter(Depression == 1)
    } else if (clinical == "noMDD") {
      input_data <- final %>%
        filter(Depression == 0)
    } else if (clinical == "ANX") {
        input_data <- final %>%
            filter(Anxiety == 1)
    } else if (clinical == "CIDI-MDD") {
        input_data <- final %>%
            filter(MDD_Incl == 1)
    }  else if (clinical == "CIDI-noMDD") {
        input_data <- final %>%
            filter(MDD_Incl == 0)
      } else {
        input_data <- final
    }
  
  #-- Create all combinations and run models
  results_all <- expand_grid(
    population = population_groups,
    outcome = outcomes,
    independent = predictors
  ) %>%
    mutate(
      results = pmap(list(population, outcome, independent), function(pop, out, indep) {
        
        # Filter for population
        input_data_pop <- input_data %>% filter(ancestry_pred == pop)
        
        # Run model with error handling
        result <- tryCatch(
          model_function(input_data_pop, indep, out),
          error = function(e) {
            message("Error: ", pop, ", ", out, ", ", indep, " - ", e$message)
            return(NULL)
          }
        )
        
        # Add metadata if successful
        if (is.data.frame(result)) {
          result %>%
            mutate(
              GeneticMarker = indep,
              ClinicalGroup = clinical,
              Population = pop,
              Outcome = out
            )
        } else {
          NULL
        }
      })
    ) %>%
    unnest(results) %>%
    dplyr::select(Population, ClinicalGroup, Outcome, GeneticMarker, N_Total, N_Cases, 
                  N_Controls, variable, odds_ratio, lower_95, upper_95, p_value)
  
  #-- Apply multiple testing correction
  results_final <- results_all %>%
    mutate(
      Bonf_P = pmin(p_value * 7, 1)
    ) %>%
      mutate(
        across(c(odds_ratio, lower_95, upper_95), ~ signif(.x, 4)),
        p_value_num = p_value,
        Bonf_P_num  = Bonf_P,
        p_value_display = case_when(
          p_value < 2.2e-16 ~ "<2.2e-16",
          TRUE ~ formatC(p_value, format = "e", digits = 2)
        ),
        Bonf_P_display = case_when(
          Bonf_P < 2.2e-16 ~ "<2.2e-16",
          TRUE ~ formatC(Bonf_P, format = "e", digits = 2)
        )
      )
  
  return(results_final)
  
}) %>%
  set_names(clinical_groups)

In [ ]:
#-- New ordinal loop (parallel structure)
ordinal_list <- map(clinical_groups, function(clinical) {
  
  cat("\n=== Ordinal:", clinical, "===\n")
  
  input_data <- if (clinical == "MDD") {
      input_data <- final %>%
        filter(Depression == 1)
    } else if (clinical == "noMDD") {
      input_data <- final %>%
        filter(Depression == 0)
    } else if (clinical == "ANX") {
        input_data <- final %>%
            filter(Anxiety == 1)
    } else if (clinical == "CIDI-MDD") {
        input_data <- final %>%
            filter(MDD_Incl == 1)
    }  else if (clinical == "CIDI-noMDD") {
        input_data <- final %>%
            filter(MDD_Incl == 0)
      } else {
        input_data <- final
    }
  
  results_all <- expand_grid(
    population  = population_groups,
    independent = predictors
  ) %>%
    mutate(
      results = pmap(list(population, independent), function(pop, indep) {
        
        input_data_pop <- input_data %>% filter(ancestry_pred == pop)
        
        result <- tryCatch(
          model_function_ordinal(input_data_pop, indep),
          error = function(e) {
            message("Error: ", pop, ", ", indep, " - ", e$message)
            return(NULL)
          }
        )
        
        if (is.data.frame(result)) {
          result %>%
            mutate(
              GeneticMarker = indep,
              ClinicalGroup = clinical,
              Population    = pop
            )
        } else {
          NULL
        }
      })
    ) %>%
    unnest(results) %>%
    filter(term_type == "coefficient") %>%
    dplyr::select(Population, ClinicalGroup, GeneticMarker, 
                  N_Total, N_Continuation, N_Switching, N_Augmentation,
                  variable, odds_ratio, lower_95, upper_95, p_value)
  
  #-- Bonferroni — scale by actual number of PGS tested per group
  results_final <- results_all %>%
    group_by(Population, ClinicalGroup) %>%
    mutate(Bonf_P = pmin(p_value * n_distinct(GeneticMarker), 1)) %>%
    ungroup() %>%
    mutate(
      across(c(odds_ratio, lower_95, upper_95), ~ signif(.x, 4)),
      p_value_num     = p_value,
      Bonf_P_num      = Bonf_P,
      p_value_display = case_when(
        p_value < 2.2e-16 ~ "<2.2e-16",
        TRUE              ~ formatC(p_value, format = "e", digits = 2)
      ),
      Bonf_P_display  = case_when(
        Bonf_P < 2.2e-16 ~ "<2.2e-16",
        TRUE             ~ formatC(Bonf_P, format = "e", digits = 2)
      )
    )
  
  return(results_final)
  
}) %>% set_names(clinical_groups)

In [ ]:
results_df <- bind_rows(all_list)
results_df <- results_df %>%
    mutate(
        GeneticMarker = case_when(
        GeneticMarker == "ADHD_01_PGS_std" ~ "ADHD",
        GeneticMarker == "Migraine_01_PGS_std" ~ "Migraine",
        GeneticMarker == "Insomnia_01_PGS_std" ~ "Insomnia",
        GeneticMarker == "MDD_04_PGS_std" ~ "Depression",
        GeneticMarker == "SCZ_03_PGS_std" ~ "Schizophrenia",
        GeneticMarker == "BIP_2025_PGS_std" ~ "Bipolar Disorder",
        GeneticMarker == "BMI_LOO_PGS_std" ~ "BMI",
        TRUE ~ NA_character_)) %>%
    mutate(variable = case_when(
        variable == "ADHD_01_PGS_std" ~ "ADHD",
        variable == "Migraine_01_PGS_std" ~ "Migraine",
        variable == "Insomnia_01_PGS_std" ~ "Insomnia",
        variable == "MDD_04_PGS_std" ~ "Depression",
        variable == "SCZ_03_PGS_std" ~ "Schizophrenia",
        variable == "BIP_2025_PGS_std" ~ "Bipolar Disorder",
        variable == "BMI_LOO_PGS_std" ~ "BMI",
        TRUE ~ variable)) %>%
    mutate(Population = case_when(
        Population == "afr" ~ "African",
        Population == "eur" ~ "European",
        Population == "amr" ~ "Latino/Mixed-American",
        TRUE ~ NA_character_)) %>%
    mutate(ClinicalGroup = case_when(
        ClinicalGroup == "Full" ~ "Full Cohort",
        ClinicalGroup == "MDD" ~ "Recorded Depression",
        ClinicalGroup == "noMDD" ~ "No Recorded Depression",
        ClinicalGroup == "ANX" ~ "Recorded Anxiety",
        TRUE ~ ClinicalGroup)) %>%
    mutate(Outcome = case_when(
        Outcome == "trd_status" ~ "TRD",
        Outcome == "combo_status" ~ "Combination",
        Outcome == "aug_status" ~ "Augmentation",
        TRUE ~ Outcome))

pop_order <- c("European", "African", "Latino/Mixed-American")
outcome_order <- c("Augmentation")

clinical_order <- c(
  "Full Cohort",
  "Recorded Depression",
  "No Recorded Depression",
  "Recorded Anxiety",
  "CIDI-MDD",
  "CIDI-noMDD"
)

results_df <- results_df %>%
  rename(PGS = GeneticMarker) %>%
  mutate(
    Population = factor(Population, levels = pop_order),
    ClinicalGroup = factor(ClinicalGroup, levels = clinical_order),
    Outcome = factor(Outcome, levels = outcome_order)
  ) %>%
  arrange(Population, ClinicalGroup, Outcome)

In [ ]:
results_df_ord <- bind_rows(ordinal_list)
results_df_ord <- results_df_ord %>%
  mutate(
    GeneticMarker = case_when(
      GeneticMarker == "ADHD_01_PGS_std"     ~ "ADHD",
      GeneticMarker == "Migraine_01_PGS_std" ~ "Migraine",
      GeneticMarker == "Insomnia_01_PGS_std" ~ "Insomnia",
      GeneticMarker == "MDD_04_PGS_std"      ~ "Depression",
      GeneticMarker == "SCZ_03_PGS_std"      ~ "Schizophrenia",
      GeneticMarker == "BIP_2025_PGS_std"    ~ "Bipolar Disorder",
      GeneticMarker == "BMI_LOO_PGS_std"     ~ "BMI",
      TRUE ~ NA_character_
    )
  ) %>%
  mutate(
    variable = case_when(
      variable == "ADHD_01_PGS_std"     ~ "ADHD",
      variable == "Migraine_01_PGS_std" ~ "Migraine",
      variable == "Insomnia_01_PGS_std" ~ "Insomnia",
      variable == "MDD_04_PGS_std"      ~ "Depression",
      variable == "SCZ_03_PGS_std"      ~ "Schizophrenia",
      variable == "BIP_2025_PGS_std"    ~ "Bipolar Disorder",
      variable == "BMI_LOO_PGS_std"     ~ "BMI",
      TRUE ~ variable
    )
  ) %>%
  mutate(
    Population = case_when(
      Population == "afr" ~ "African",
      Population == "eur" ~ "European",
      Population == "amr" ~ "Latino/Mixed-American",
      TRUE ~ NA_character_
    )
  ) %>%
  mutate(
    ClinicalGroup = case_when(
      ClinicalGroup == "Full"  ~ "Full Cohort",
      ClinicalGroup == "MDD"   ~ "Recorded Depression",
      ClinicalGroup == "noMDD" ~ "No Recorded Depression",
      ClinicalGroup == "ANX"   ~ "Recorded Anxiety",
      TRUE ~ ClinicalGroup
    )
  )

pop_order <- c("European", "African", "Latino/Mixed-American")
clinical_order <- c(
  "Full Cohort",
  "Recorded Depression",
  "No Recorded Depression",
  "Recorded Anxiety",
  "CIDI-MDD",
  "CIDI-noMDD"
)

results_df_ord <- results_df_ord %>%
  rename(PGS = GeneticMarker) %>%
  mutate(
    Population    = factor(Population, levels = pop_order),
    ClinicalGroup = factor(ClinicalGroup, levels = clinical_order)
  ) %>%
  arrange(Population, ClinicalGroup, PGS)
results_df_ord

In [ ]:
test_data <- final %>% filter(ancestry_pred == "eur" & MDD_Incl == 1) 
test_data$outcome_ord <- factor(test_data$aug_switch_cont,
                                levels = c("Continuation", "Switching", "Augmentation"),
                                ordered = TRUE)

m_anx <- MASS::polr(outcome_ord ~ age_at_first_AD + sex_at_birth + 
                    Income_quantitative + Education_level + MDD_04_PGS_std,
                    data = test_data, Hess = TRUE)
brant::brant(m_anx)

In [ ]:
results_df_ord <- results_df_ord  %>%
  filter(Population == "European" | !ClinicalGroup %in% c("CIDI-MDD", "CIDI-noMDD"))
aug_df <- results_df %>%
    filter(Outcome == "Augmentation") %>%
    filter(Population == "European" | !ClinicalGroup %in% c("CIDI-MDD", "CIDI-noMDD"))
trd_df <- results_df %>%
    filter(Outcome == "TRD") %>%
    filter(Population == "European" | !ClinicalGroup %in% c("CIDI-MDD", "CIDI-noMDD"))

wb <- loadWorkbook("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx")
removeWorksheet(wb, "Table4")
addWorksheet(wb, "Table4")
writeData(wb, "Table4", aug_df)
removeWorksheet(wb, "Table5")
addWorksheet(wb, "Table5")
writeData(wb, "Table5", results_df_ord)
saveWorkbook(wb, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx", overwrite=TRUE)


In [ ]:
#-- Radial plot
#-- results
complexity <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx", sheet = "Table4") 
complexity <- complexity %>%
  filter(!variable %in% c("(Intercept)", "sex_at_birthMale", "age_at_first_AD")) %>%
  filter(!str_detect(variable, "PC")) %>%
  filter(!str_detect(variable, "Income_quantitative")) %>%
  filter(!str_detect(variable, "Education_level")) %>%
  filter(ClinicalGroup == "Recorded Depression") %>%
  filter(Population == "European") %>%
  select(Outcome, PGS, odds_ratio)

switching <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx", sheet = "Table3") %>%
  filter(Population == "European") %>%
  filter(ClinicalGroup == "Recorded Depression") %>%
  filter(variable == PGS) %>%
  filter(Outcome == "Switching vs Continuation") %>%
  select(Outcome, PGS, odds_ratio) %>%
  mutate(Outcome = "Switching")

dat <- rbind(complexity, switching) %>%
    mutate(odds_ratio = ifelse(odds_ratio < 1, 1, odds_ratio))
summary(dat$odds_ratio)

library(dplyr)
library(tidyr)
library(fmsb)

# dat must have columns: Outcome, PGS, odds_ratio
pgs_order <- unique(dat$PGS)

# long -> wide
wide_dat <- dat %>%
  mutate(PGS = factor(PGS, levels = pgs_order)) %>%
  select(Outcome, PGS, odds_ratio) %>%
  pivot_wider(names_from = PGS, values_from = odds_ratio) %>%
  as.data.frame()

rownames(wide_dat) <- wide_dat$Outcome
wide_dat$Outcome <- NULL

# fill missing with OR = 1
wide_dat[is.na(wide_dat)] <- 1

# transform to log OR
wide_log <- log(wide_dat)

# set limits dynamically from data
upper <- max(wide_log, na.rm = TRUE)
lower <- min(0, min(wide_log, na.rm = TRUE))

wide_plot <- wide_log

# fmsb format: first row max, second row min
radar_dat <- rbind(
  rep(upper, ncol(wide_plot)),
  rep(lower, ncol(wide_plot)),
  wide_plot
)

# colors
border_cols <- c("#F8766D", "#7CAE00", "#00BFC4", "#C77CFF")[seq_len(nrow(wide_plot))]
fill_cols <- adjustcolor(border_cols, alpha.f = 0.20)

# generate axis labels dynamically in OR units
axis_breaks <- seq(lower, upper, length.out = 4)
axis_labs <- as.character(round(exp(axis_breaks), 2))

png("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Main_AoU_Radar_Plot.png",
    width = 10, height = 7, units = "in", res = 300)

par(mar = c(10, 2, 4, 2), bg = "white")
radarchart(
  radar_dat,
  axistype = 1,
  seg = 3,
  caxislabels = axis_labs,
  pcol = border_cols,
  pfcol = fill_cols,
  plwd = 2,
  plty = 1,
  vlabels = colnames(wide_plot),
  vlcex = 1.1,
  cglcol = "grey80",
  cglty = 1,
  cglwd = 0.8,
  axislabcol = "grey40",
  title = "AoU individuals with Recorded Depression\nPGS Odds ratio by Outcome (log scale)"
)
legend(
  x = -0.3, y = -1.4,
  legend = rownames(wide_plot),
  col = border_cols,
  pt.bg = fill_cols,
  pch = 16,
  pt.cex = 1.5,
  bty = "n",
  cex = 1.1,
  horiz = FALSE,
  ncol = 2,
  xpd = TRUE
)

dev.off()

In [ ]:
#-- Across all clinical subsets within Europeans (Augmentation therapy)
library(readxl)
library(stringr)
library(cowplot)

#-- Main plot
processed_data <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx", sheet = "Table4") %>%
  filter(!variable %in% c("(Intercept)", "sex_at_birthMale", "age_at_first_AD")) %>%
  filter(!str_detect(variable, "PC")) %>%
  filter(!str_detect(variable, "Income_quantitative")) %>%
  filter(!str_detect(variable, "Education_level")) %>%
  filter(Population == "European") %>%
  filter(ClinicalGroup != "Recorded Anxiety") %>%
  mutate(SigFill = if_else(Bonf_P_num < 0.05, ClinicalGroup, "NotSig")) %>%
    mutate(
    Outcome = "All of Us\nRecorded Depression:\nN=6,704 (21%)\nNo Recorded Depression:\nN=9,145 (3.8%)\nFull Cohort: N=15,849 (11%)\nCIDI-noMDD: N=1479(3%)\nCIDI-MDD: N=2027 (23%)")

order <- processed_data %>%
  filter(ClinicalGroup == "Recorded Depression") %>%
  arrange(odds_ratio) %>%
  pull(PGS) %>%
  unique()

processed_data$PGS <- factor(
  processed_data$PGS, 
  levels = order
)

processed_data$ClinicalGroup = factor(processed_data$ClinicalGroup, levels = rev(c("Recorded Depression", "Full Cohort", "No Recorded Depression", "CIDI-MDD", "CIDI-noMDD")))

pos <- position_dodge(width = 0.5)
fill_colors <- c(
  "No Recorded Depression" = "#2CA58D80",
  "Recorded Depression"    = "#D11149",
  "Full Cohort"   = "#6610F280",
   "CIDI-noMDD"       = "#2E6F9580",
  "CIDI-MDD"      = "#E0A526"
)

# Plot A
pA_AoU <- ggplot(
    processed_data,
    aes(x = odds_ratio, y = PGS, group = ClinicalGroup)
  ) +
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95, colour = ClinicalGroup), height = 0, position = pos) +
  geom_point(aes(fill = SigFill, colour = ClinicalGroup), shape = 21, size = 4, stroke = 0.5, position = pos) +
  scale_color_manual(values = fill_colors) +
  facet_wrap(Outcome ~ .) +
  scale_fill_manual(values = c(fill_colors, NotSig = "white"), guide = "none") +
  labs(x = "Odds ratio with 95% CI,\nRef = Continuation", y = "PGS", color = "Diagnostic\nSubset") +
  theme_minimal() +
  theme(
  axis.title        = element_text(color = "black", face = "bold"),
  legend.position   = "left",
  strip.text        = element_text(face = "bold", size = 11),
  strip.background  = element_rect(fill = "white", color = "black"),
  axis.text        = element_text(color = "black"),
  axis.text.y      = element_text(size = 13, color = "black"),
  panel.background  = element_rect(fill = "white", colour = NA),
  plot.background   = element_rect(fill = "white", colour = NA),
  plot.margin = margin(t = 20, r = 5, b = 5, l = 5),
  plot.title        = element_text(face = "bold")
)

#-- Main plot for LL
processed_data <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL_noANX.xlsx", sheet = "Table4") %>%
  filter(!variable %in% c("(Intercept)", "sex_at_birth", "age_at_first_AD")) %>%
  filter(!str_detect(variable, "PC")) %>%
  filter(!str_detect(variable, "Income_level")) %>%
  filter(!str_detect(variable, "Education_level")) %>%
  filter(Population == "European") %>%
  filter(ClinicalGroup != "Recorded Anxiety") %>%
  filter(Outcome == "Augmentation") %>%
  mutate(SigFill = if_else(Bonf_P < 0.05, ClinicalGroup, "NotSig")) %>%
    mutate(
    Outcome = "Lifelines\nRecorded Depression:\nN=1,831 (12%)\nNo Recorded Depression:\nN=2,741 (5.8%)\nFull Cohort: N=4,623 (8.3%)")


processed_data$PGS <- factor(
  processed_data$PGS, 
  levels = order
)

processed_data$ClinicalGroup = factor(processed_data$ClinicalGroup, levels = rev(c("Recorded Depression", "Full Cohort", "No Recorded Depression")))

fill_colors <- c(
  "No Recorded Depression" = "#2CA58D80",
  "Recorded Depression"    = "#D11149",
  "Full Cohort"   = "#6610F280"
)

# Plot A
pA_LL <- ggplot(
    processed_data,
    aes(x = odds_ratio, y = PGS, group = ClinicalGroup)
  ) +
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95, colour = ClinicalGroup), height = 0, position = pos) +
  geom_point(aes(fill = SigFill, colour = ClinicalGroup), shape = 21, size = 4, stroke = 0.5, position = pos) +
  facet_wrap(Outcome ~ .) +
  scale_color_manual(values = fill_colors) +
  scale_fill_manual(values = c(fill_colors, NotSig = "white"), guide = "none") +
  labs(x = "Odds ratio with 95% CI,\nRef = Continuation", y = "PGS", color = "Diagnostic\nSubset") +
  theme_minimal() +
  theme(
  axis.title        = element_text(color = "black", face = "bold"),
  legend.position   = "none",
  plot.title        = element_text(face = "bold"),
  strip.text        = element_text(face = "bold", size = 11),
  strip.background  = element_rect(fill = "white", color = "black"),
  axis.text        = element_text(color = "black"),
  axis.text.y      = element_text(size = 13, color = "black"),
  panel.background  = element_rect(fill = "white", colour = NA),
  plot.background   = element_rect(fill = "white", colour = NA),
  plot.margin       = margin(t = 20, r = 5, b = 5, l = 5),
  legend.title      = element_text(face = "bold")
)


# Combine
p_combined <- plot_grid(pA_AoU, pA_LL, ncol = 2, rel_widths = c(1, 0.55))
p_combined

In [ ]:
library(ggplotify)

# Wrap the radar in a function (drop the png()/dev.off() wrapping)
draw_radar <- function() {
  par(mar = c(4, 2, 2, 2), oma = c(0, 0, 2, 0), bg = "white")
  radarchart(
    radar_dat,
    axistype     = 1,
    seg          = 3,
    caxislabels  = axis_labs,
    pcol         = border_cols,
    pfcol        = fill_cols,
    plwd         = 2,
    plty         = 1,
    vlabels      = colnames(wide_plot),
    vlcex        = 0.8,
    cglcol       = "grey80",
    cglty        = 1,
    cglwd        = 0.8,
    axislabcol   = "grey40",
    title        = "AoU (with Recorded Depression)\nPGS Odds ratio by Outcome (log scale)"
  )
  legend(
    x = -0.3, y = -1.4,
    legend = rownames(wide_plot),
    col    = border_cols,
    pt.bg  = fill_cols,
    pch    = 16,
    pt.cex = 0.7,
    bty    = "n",
    cex    = 1,
    horiz  = FALSE,
    ncol   = 1,
    xpd    = TRUE
  )
}

radar_gg <- as.ggplot(draw_radar)

# Stack: forest row on top, radar row below
top_row <- plot_grid(pA_AoU, pA_LL, ncol = 2, rel_widths = c(1, 0.65))

radar_row <- plot_grid(
  radar_gg, NULL,
  ncol = 2,
  rel_widths = c(1, 0.8)
)


In [ ]:
# Centered subtitle strip for panel A
titleA <- ggdraw() +
  draw_label("Augmentation", fontface = "bold", size = 16, x = 0.5, hjust = 0.5)

# Panel A = subtitle strip + forest row
panelA <- plot_grid(
  titleA, top_row,
  ncol        = 1,
  rel_heights = c(0.08, 1)
)

p_final <- plot_grid(
  panelA,
  radar_row,
  ncol        = 1,
  rel_heights = c(1.08, 1),   # bumped slightly to account for the title strip
  labels      = c("A", "B"),
  label_size  = 16
)
p_final

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Main_TreatmentComplexity.png",
       plot   = p_final,
       width  = 12,
       height = 11,
       device = "png",
       units  = "in")

In [ ]:
library(readxl)
library(stringr)
library(dplyr)
library(tidyverse)
library(scales)

aou_path <- "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx"

# ---- prep: European only, keep PGS term, key significance on diagnostic subset ----
prep_ord <- function(raw, p_col = "Bonf_P_num") {
  raw %>%
    mutate(across(c(variable, PGS), str_trim)) %>%
    filter(!variable %in% c("(Intercept)", "sex_at_birthMale", "sex_at_birth", "age_at_first_AD")) %>%
    filter(!str_detect(variable, "PC")) %>%
    filter(!str_detect(variable, "Income")) %>%
    filter(!str_detect(variable, "Education_level")) %>%
    filter(Population == "European") %>%
    filter(ClinicalGroup != "Recorded Anxiety") %>%
    filter(variable == PGS) %>%
    mutate(
      odds_ratio    = as.numeric(odds_ratio),
      lower_95      = as.numeric(lower_95),
      upper_95      = as.numeric(upper_95),
      SigFill       = if_else(.data[[p_col]] < 0.05, ClinicalGroup, "NotSig")
    )
}

ord_aou <- prep_ord(read_excel(aou_path, sheet = "Table5"), p_col = "Bonf_P")

# ---- dataset + per-subset N for the facet strip ----
make_strip <- function(d, dataset) {
  ns <- d %>%
    distinct(ClinicalGroup, N_Total) %>%
    mutate(ClinicalGroup = factor(ClinicalGroup,
                                  levels = c("Recorded Depression", "No Recorded Depression",
                                             "Full Cohort", "CIDI-MDD", "CIDI-noMDD"))) %>%
    arrange(ClinicalGroup)

  paste0(dataset, "\n",
         paste(sprintf("%s: N=%s", ns$ClinicalGroup, format(ns$N_Total, big.mark = ",")),
               collapse = "\n"))
}

ord_aou <- ord_aou %>% mutate(Outcome = make_strip(ord_aou, "All of Us"))

# ---- PGS order (European, Depression subset, by OR) ----
order <- ord_aou %>%
  filter(ClinicalGroup == "Recorded Depression") %>%
  arrange(odds_ratio) %>% pull(PGS) %>% unique()

ord_aou$PGS           <- factor(ord_aou$PGS, levels = order)
ord_aou$ClinicalGroup <- factor(ord_aou$ClinicalGroup,
                                levels = rev(c("Recorded Depression", "Full Cohort", "No Recorded Depression", "CIDI-MDD", "CIDI-noMDD")))

# ---- palettes: solid rings, fade everything except Depression for the fills ----
fill_colors <- c(
  "No Recorded Depression" = "#2CA58D80",
  "Recorded Depression"    = "#D11149",
  "Full Cohort"            = "#6610F280",
  "CIDI-noMDD"             = "#2E6F9580",
  "CIDI-MDD"               = "#E0A526"
)
fill_faded  <- fill_colors
non_dep     <- setdiff(names(fill_colors), "Recorded Depression")
fill_faded[non_dep] <- alpha(fill_colors[non_dep], 0.5)

pos   <- position_dodge(width = 0.5)
x_lab <- "Cumulative OR per 1-SD PGS (95% CI), ref = 1\nhigher \u2192 more complex outcome\n(Continuation \u2192 Switching \u2192 Augmentation)"

pOrd_AoU <- ggplot(ord_aou, aes(x = odds_ratio, y = PGS, group = ClinicalGroup)) +
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95, colour = ClinicalGroup), height = 0, position = pos) +
  geom_point(aes(fill = SigFill, colour = ClinicalGroup),
             shape = 21, size = 4, stroke = 0.5, position = pos) +
  scale_color_manual(values = fill_colors) +
  scale_fill_manual(values = c(fill_faded, NotSig = "white"), guide = "none") +
  guides(color = guide_legend(nrow = 3)) +
  facet_wrap(Outcome ~ .) +
  labs(x = x_lab, y = "PGS", color = "Diagnostic\nSubset") +
  theme_minimal() +
  theme(
    axis.title       = element_text(color = "black", face = "bold"),
    legend.position  = "bottom",
    plot.title       = element_text(face = "bold"),
    strip.text       = element_text(face = "bold", size = 11),
    strip.background = element_rect(fill = "white", color = "black"),
    axis.text        = element_text(color = "black"),
    axis.text.y      = element_text(size = 13, color = "black"),
    panel.background = element_rect(fill = "white", colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    plot.margin      = margin(t = 20, r = 5, b = 5, l = 5)
  )

pOrd_AoU

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_Ordinal_European.png",
       plot   = pOrd_AoU,
       width  = 8,
       height = 8,
       device = "png",
       units  = "in")

In [ ]:
library(readxl)
library(dplyr)
library(tidyverse)
library(stringr)
library(scales)

aou_path <- "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx"

processed_data <- read_excel(aou_path, sheet = "Table4") %>%
  mutate(across(c(variable, PGS), str_trim)) %>%
  filter(!variable %in% c("(Intercept)", "sex_at_birthMale", "age_at_first_AD")) %>%
  filter(!str_detect(variable, "PC")) %>%
  filter(!str_detect(variable, "Income_quantitative")) %>%
  filter(!str_detect(variable, "Education_level")) %>%
  filter(!ClinicalGroup %in% c("Recorded Anxiety", "CIDI-MDD", "CIDI-noMDD")) %>%
  filter(variable == PGS) %>%                              # keep the PGS term only
  mutate(
    odds_ratio    = as.numeric(odds_ratio),
    lower_95      = as.numeric(lower_95),
    upper_95      = as.numeric(upper_95),
    SigFill       = if_else(Bonf_P_num < 0.05, Population, "NotSig")   # sig keyed on ancestry
  )
unique(processed_data$ClinicalGroup)
#-- Ancestry order (largest N first); controls dodge + legend order
anc_order <- processed_data %>%
  group_by(Population) %>% summarise(N = max(N_Total), .groups = "drop") %>%
  arrange(desc(N)) %>% pull(Population) %>% as.character()
processed_data$Population <- factor(processed_data$Population, levels = rev(anc_order))

#-- Named palette (matched by name, not position)
fill_colors <- c(
  "European"              = "#1B7837",  # forest green
  "African"               = "#762A83",  # purple
  "Latino/Mixed-American" = "#E08214"   # amber/orange
)

#-- Faded copy for the interiors: keep European solid, soften the rest
fill_faded <- fill_colors
non_eur    <- setdiff(names(fill_colors), "European")
fill_faded[non_eur] <- alpha(fill_colors[non_eur], 0.5)

#-- PGS order from European, Depression subset (reference)
order <- processed_data %>%
  filter(ClinicalGroup == "Recorded Depression", Population == "European") %>%
  arrange(odds_ratio) %>% pull(PGS) %>% unique()
processed_data$PGS <- factor(processed_data$PGS, levels = order)

cg_levels <- intersect(c("Recorded Depression", "Full Cohort", "No Recorded Depression"), unique(processed_data$ClinicalGroup))
processed_data$ClinicalGroup <- factor(processed_data$ClinicalGroup, levels = cg_levels)

#-- Facet header: ClinicalGroup + per-ancestry N (+ augmentation rate)
make_facet_label <- function(d) {
  ns <- d %>% select(Population, N_Total, N_Cases) %>% distinct() %>% arrange(Population)
  paste0(as.character(unique(d$ClinicalGroup)), "\n",
         paste(sprintf("%s: N=%s (%s%%)", ns$Population,
                       format(ns$N_Total, big.mark = ","),
                       round((ns$N_Cases / ns$N_Total) * 100, 1)),
               collapse = "\n"))
}
processed_data <- processed_data %>%
  group_by(ClinicalGroup) %>%
  mutate(FacetLabel = make_facet_label(cur_data_all())) %>%
  ungroup()
lab_order <- processed_data %>% distinct(ClinicalGroup, FacetLabel) %>%
  arrange(ClinicalGroup) %>% pull(FacetLabel)
processed_data$FacetLabel <- factor(processed_data$FacetLabel, levels = lab_order)

pos <- position_dodge(width = 0.6)

p_aug_anc <- ggplot(processed_data, aes(x = odds_ratio, y = PGS, group = Population)) +
  geom_vline(xintercept = 1, linetype = "dashed") +
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95, colour = Population), height = 0, position = pos) +
  geom_point(aes(fill = SigFill, colour = Population),
             shape = 21, size = 4, stroke = 0.5, position = pos) +
  scale_color_manual(values = fill_colors) +
  scale_fill_manual(values = c(fill_faded, NotSig = "white"), guide = "none") +
  facet_wrap(~ FacetLabel, nrow = 2, scales = "free_x") +
  labs(x = "Odds ratio with 95% CI,\nRef = Continuation",
       y = "PGS", title = "AoU — Augmentation across ancestries",
       color = "Ancestry") +
  theme_minimal() +
  theme(
    axis.title       = element_text(color = "black", face = "bold"),
    plot.title       = element_text(face = "bold", size = 14, hjust = 0.5),
    strip.text       = element_text(size = 11),
    strip.background = element_rect(fill = "white", color = "black"),
    axis.text        = element_text(color = "black"),
    axis.text.y      = element_text(size = 13, color = "black"),
    panel.background = element_rect(fill = "white", colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    plot.margin      = margin(t = 20, r = 5, b = 5, l = 5),
    legend.title     = element_text(face = "bold"),
    legend.position  = "bottom"
  )
p_aug_anc

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_Aug_Ancestries.png",
       plot   = p_aug_anc,
       width  = 9,
       height = 8,
       device = "png",
       units  = "in")